# M04A — Parameter Estimation: MLE, MAP & EM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tulip-lab/pattern-classification-lab/blob/develop/M04-Parameter-Estimation/M04A-Parameter-Estimation-MLE-MAP-EM.ipynb)

**Pattern Classification practical · approximately 2 hours · M04 · CLO2, CLO3**

This is the canonical student-facing practical for session `M04A`.
It was newly authored for the Pattern Classification Lab using concepts and
public migration references from `tulip-lab/flip01`: `F1A-02-EM.ipynb`, `F1A-00-Bayesian.ipynb`. No legacy
notebook cells or outputs are copied verbatim. The implementation uses current
public APIs and synthetic or scikit-learn-bundled data.

**Licence:** code cells are available under MPL-2.0; original narrative teaching
content is available under CC BY-NC-SA 4.0. Third-party works and datasets retain
their own terms. See the repository `LICENSE`, `CONTENT-LICENSE.md`, and
`NOTICE.md`.

## Overview

Parameter estimates connect data to probabilistic classifiers. You will contrast likelihood-only MLE with prior-regularised MAP, then use EM when the component labels of a mixture are unobserved.

Work through the guided cells first, then complete the challenge in your own
copy. Keep your interpretation beside the code: a result without an argument
is not complete evidence.

## Learning Objectives

- Estimate Gaussian parameters by maximum likelihood.
- Compute a normal-prior MAP estimate and explain shrinkage.
- Fit a Gaussian mixture with EM and interpret responsibilities, weights, and convergence.
- Evaluate sensitivity to sample size and initialisation.

## Expected Output and Evidence

Retain the following evidence in your executed notebook:

- MLE and MAP estimates with an explanation of shrinkage
- an executed Gaussian-mixture fit with convergence evidence
- an initialisation-sensitivity comparison and reflection

Do not include credentials, private data, or another learner's work.

## Prerequisites

- Gaussian distributions
- Log-likelihood
- Basic Bayes' rule
- Array operations

## Session Plan

| Time | Activity | Evidence |
|---:|---|---|
| 0–10 min | Orientation and diagnostic prompt | Initial prediction or sketch |
| 10–30 min | Background concepts and setup | Concept notes and reproducible environment |
| 30–75 min | Guided practice | Executed analysis with interpretations |
| 75–105 min | Student challenge | Independent model choice and evidence |
| 105–115 min | Testing and debugging | Passing checks and resolved issues |
| 115–120 min | Submission and reflection | Concise evidence-based reflection |

## Background Concepts

For independent Gaussian observations with unknown mean, the MLE is the sample mean. With a Gaussian prior on the mean, MAP balances data precision and prior precision, producing shrinkage toward the prior mean.

In a mixture model, component membership is latent. EM alternates an **E-step** (soft responsibilities for components) and an **M-step** (parameter updates). The data log-likelihood should not decrease, although different initialisations can reach different local optima.

## Accessibility and Responsible Use

Every visual result in the guided path is paired with a printed numeric summary. Do not rely on colour alone in your challenge evidence: add labels, line styles, markers, or a compact table as appropriate. Add concise alt text when exporting figures for another format. Use only public, approved, or synthetic data.

Follow the collaboration, acknowledgement, and AI-use rules published by your current offering; this institution-neutral practical does not define those rules.

## Setup

Run the next cell before starting. All datasets are bundled with scikit-learn or generated locally; no download is required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.mixture import GaussianMixture

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
plt.style.use("seaborn-v0_8-whitegrid")

## Guided Practice

Follow the sequence and pause at each interpretation prompt before moving on.

### 1. MLE for a Gaussian mean and variance

Use `ddof=0` for the Gaussian maximum-likelihood variance; `ddof=1` is the common unbiased sample estimate.

In [ ]:
observations = rng.normal(loc=3.0, scale=1.5, size=40)
mle_mean = observations.mean()
mle_variance = np.mean((observations - mle_mean) ** 2)
print(f"MLE mean={mle_mean:.3f}, MLE variance={mle_variance:.3f}")

### 2. MAP estimate with a Gaussian prior

The observation variance is treated as known here. Change `prior_sd` and predict the direction of movement before rerunning.

In [ ]:
known_variance = 1.5 ** 2
prior_mean, prior_sd = 0.0, 1.0
n = len(observations)
map_mean = (
    observations.sum() / known_variance + prior_mean / prior_sd**2
) / (n / known_variance + 1 / prior_sd**2)
print(f"Prior mean={prior_mean:.3f}, MAP mean={map_mean:.3f}, MLE mean={mle_mean:.3f}")

### 3. Create an unlabeled mixture

The histogram reveals overlapping groups, but component labels are not supplied to the estimator.

In [ ]:
component = rng.choice([0, 1], size=600, p=[0.35, 0.65])
mixture_data = np.where(
    component == 0,
    rng.normal(-2.0, 0.7, size=600),
    rng.normal(2.5, 1.0, size=600),
).reshape(-1, 1)
plt.hist(mixture_data[:, 0], bins=40, density=True, alpha=0.65)
plt.title("Unlabelled observations from a two-component mixture")
plt.show()

### 4. Fit EM and inspect soft assignments

`predict_proba` returns responsibilities. Component numbers are arbitrary, so sort components by their estimated means before interpreting them.

In [ ]:
gmm = GaussianMixture(n_components=2, n_init=10, random_state=RANDOM_STATE)
gmm.fit(mixture_data)
order = np.argsort(gmm.means_.ravel())
responsibilities = gmm.predict_proba(mixture_data)[:, order]
print("converged:", gmm.converged_, "iterations:", gmm.n_iter_)
print("means:", gmm.means_.ravel()[order].round(3))
print("weights:", gmm.weights_[order].round(3))
print("first five responsibilities:\n", responsibilities[:5].round(3))

### 5. Overlay the fitted density

The mixture density is the weighted sum of component densities.

In [ ]:
grid = np.linspace(-5, 6, 500).reshape(-1, 1)
density = np.exp(gmm.score_samples(grid))
plt.hist(mixture_data[:, 0], bins=40, density=True, alpha=0.45, label="data")
plt.plot(grid[:, 0], density, color="black", linewidth=2, label="fitted mixture")
plt.legend()
plt.title("EM Gaussian-mixture fit")
plt.show()

## Student Challenge

Investigate EM sensitivity. Fit one model for each seed from 0 to 9 with `n_init=1`. Record convergence, iterations, and final average log-likelihood (`lower_bound_`). Explain whether the initialisation changed the substantive conclusion.

In [ ]:
initialisation_results = []
for seed in range(10):
    candidate = GaussianMixture(n_components=2, n_init=1, random_state=seed)
    candidate.fit(mixture_data)
    # TODO: append seed, convergence, iterations, lower_bound, sorted means, and weights.
    pass

initialisation_results[:3]

### Challenge success criteria

- All ten fits use the same data and one initialisation each.
- Results include objective value and convergence evidence.
- Component label-switching is handled before comparing means.
- The conclusion distinguishes local optimisation from sampling uncertainty.

## Testing and Debugging

Run these checks after the guided practice. If one fails, inspect shapes, label order, random seeds, and whether preprocessing was fitted only on training data.

In [ ]:
# Exercise missing-information behaviour.
try:
    GaussianMixture(n_components=2, random_state=RANDOM_STATE).fit(
        np.array([[0.0], [np.nan], [1.0]])
    )
except ValueError as error:
    print("Expected missing-value failure:", str(error).split("\n")[0])
else:
    raise AssertionError("GaussianMixture should reject NaN input.")

assert len(mixture_data) > gmm.n_components
print("Edge check: the sample count exceeds the number of mixture components.")

In [ ]:
assert np.isfinite(mle_mean) and mle_variance > 0
assert prior_mean < map_mean < mle_mean
assert gmm.converged_
assert np.isclose(gmm.weights_.sum(), 1.0)
assert np.allclose(responsibilities.sum(axis=1), 1.0)
print("M04A guided-practice checks passed.")

## Extension Task

Reduce the mixture to 40 observations and repeat the MLE/MAP comparison for each component using the EM responsibilities as fractional counts. Discuss where the prior has the greatest influence.

## Submission and Reflection

Submit your completed notebook (or an HTML export) with outputs visible. Include: 

1. the challenge code and result;
2. one figure or compact results table;
3. a 150–250 word reflection answering: **What changed, why did it change, and what evidence supports your explanation?**

Follow your offering's published collaboration, acknowledgement, and AI-use rules. You remain responsible for checking and explaining all submitted work.

## References

- Sources: `F1A-02-EM.ipynb` and selected estimation ideas from `F1A-00-Bayesian.ipynb`.
- Bishop, *Pattern Recognition and Machine Learning*, Chapter 9.
- scikit-learn API: [GaussianMixture](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html)